<a href="https://colab.research.google.com/github/SRXKaiser/analitica-de-datos/blob/main/Practicas%20U3/Karely/4-SVD_Sistema_Recomedacion_23111441.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



**Nombre y No control:**

* Karely Perez Vargas
* 23111441


In [ ]:
#--------------------------------------------------------------------
# Incluye aquí todas las librerías y paquetes que requieras.
import pandas as pd
import numpy as np
import seaborn as sns


#-------------------------------------------------------------------

#Ejercicio
from sklearn.decomposition import TruncatedSVD

### **Liga de Datos de la UCI "Restaurant & consumer data".**
### **Descarga el archivo "restaurant+consumer+data.zip" y de ahí necesitaremos los archivos "rating_final.csv" y "geoplaces2.csv".**

https://archive.ics.uci.edu/dataset/232/restaurant+consumer+data



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Descarga los archivos de la página de la UCI a partir de los cuales generaremos
# nuestras matrices de utilidad.

data1 = pd.read_csv("/content/drive/MyDrive/Analítica de datos/rating_final.csv", header='infer', sep=",")
data2 = pd.read_csv("/content/drive/MyDrive/Analítica de datos/geoplaces2.csv", header='infer',  encoding='latin-1')

print(data1.shape, data2.shape)


(1161, 5) (130, 21)


**NOTA: Recuerda que cada vez que se te pida incluir líneas de código, podrás incluir 1 o más líneas, pero siempre en la sección donde se te indica hacerlo.**

## **Ejercicio - 1**

### **Explica cuál es el propósito de usar el argumento "latin-1" al cargar el segundo archivo.**

++++++++ Inicia la sección de agregar texto: +++++++++++

El parametro encoding es el estándar que define cómo se traducen los caracteres de texto (letras con tildes, eñes, símbolos) a bytes en la memoria del ordenador.
Por prederteminado, utiliza UTF-8, que es un sistema de codificación de caracteres de longitud variable (de 1 a 4 bytes) diseñado para representar cualquier carácter del estándar Unicode.
EL proposito de usar el argumento latin-1, es admitir la mayoría de los idiomas de Europa occidental y es compatible con ASCII.

++++++++ Termina la sección de agregar texto. +++++++++++

In [ ]:
# Del primer archivo obtenemos una matriz con 3 evaluaciones de los restaurantes:
# general, comida y servicio.
# Las evaluaciones pueden ser 0, 1 o 2, siendo 0 la menor calificación y 2 la
# mayor calificación:

data1.head()

,userID,placeID,rating,food_rating,service_rating
0,U1077,135085,2,2,2
1,U1077,135038,2,2,1
2,U1077,132825,2,2,2
3,U1077,135060,1,2,2
4,U1068,135104,1,1,2


In [ ]:
# Del segundo archivo obtenemos información diversa de cada restaurante:

data2.head(2)

,placeID,latitude,longitude,the_geom_meter,name,address,city,state,country,fax,...,alcohol,smoking_area,dress_code,accessibility,price,url,Rambience,franchise,area,other_services
0,134999,18.915421,-99.184871,0101000020957F000088568DE356715AC138C0A525FC46...,Kiku Cuernavaca,Revolucion,Cuernavaca,Morelos,Mexico,?,...,No_Alcohol_Served,none,informal,no_accessibility,medium,kikucuernavaca.com.mx,familiar,f,closed,none
1,132825,22.147392,-100.983092,0101000020957F00001AD016568C4858C1243261274BA5...,puesto de tacos,esquina santos degollado y leon guzman,s.l.p.,s.l.p.,mexico,?,...,No_Alcohol_Served,none,informal,completely,low,?,familiar,f,open,none


In [ ]:
# De data1 no requeriremos "rating" y de data2 solo necesitamos "placeID" y "name":
# Definimos la lista y matrices con los factores que necesitamos:
# ************* Inlcuye aquí tu código:*****************************
lista_data1 = data1[["userID", "placeID", "food_rating", "service_rating"]]
lista_data2 = data2[["placeID", "name"]]
# *********** Aquí termina la sección de agregar código *************
data1a = lista_data1
data2a = lista_data2

## **Ejercicio - 2**

In [ ]:
# Define el DataFrame que conjunte la información de las dos DataFrame data1a
# y data2a en uno solo, a través de la columna común "placeID" y que
# llamaremos "df_combinado":

# ************* Inlcuye aquí tu código:*****************************
#utilice  merge()
df_combinado =pd.merge(data1a, data2a, on="placeID")


# *********** Aquí termina la sección de agregar código *************

# Despleguemos la dimensión y los primeros renglones de este DataFrame:

print(df_combinado.shape)
df_combinado.head()

(1161, 5)


,userID,placeID,food_rating,service_rating,name
0,U1077,135085,2,2,Tortas Locas Hipocampo
1,U1077,135038,2,1,Restaurant la Chalita
2,U1077,132825,2,2,puesto de tacos
3,U1077,135060,2,2,Restaurante Marisco Sam
4,U1068,135104,1,2,vips


## **Ejercicio - 3**

In [ ]:
#    Define la matriz de utilidad cuyos renglones sean los nombres de los
#    restaurantes, las columnas los IDs de los usuarios y las entradas la
#    evaluación de la comida (food_rating). La llamaremos "UtMx_food".


# ************* Inlcuye aquí tu código:*****************************


UtMx_food = df_combinado.pivot_table(values="food_rating", index="name", columns="userID", fill_value=0)


# *********** Aquí termina la sección de agregar código *************


print('Dimensión de la matriz de Utilidad sobre la evaluación de la comida:')
print('(restaurantes, usuarios) =', (UtMx_food.shape))
UtMx_food

Dimensión de la matriz de Utilidad sobre la evaluación de la comida:
(restaurantes, usuarios) = (129, 138)


userID,U1001,U1002,U1003,U1004,U1005,U1006,U1007,U1008,U1009,U1010,...,U1129,U1130,U1131,U1132,U1133,U1134,U1135,U1136,U1137,U1138
name,,,,,,,,,,,,,,,,,,,,,
Abondance Restaurante Bar,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Arrachela Grill,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cabana Huasteca,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
Cafe Chaires,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cafeteria cenidet,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tacos de barbacoa enfrente del Tec,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
tacos de la estacion,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
tacos los volcanes,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0


## **Ejercicio - 4**

### **En la Factorización SVD la cantidad de valores singulares será menor o igual al menor valor de los renglones o columnas de la matriz de utilidad UtMx.**

### **La factorización SVD de una matriz $A$ tiene la forma:**

$A_{m\times n} = U_{m\times m}\Sigma_{m\times n}V_{n\times n}^T$

In [ ]:
# a. Aplica la transformación SVD para obtener la matriz de las variables
#    latentes de los restaurantes, en relación a la manera en que los usuarios
#    evaluaron la comida. Deberás obtener la factorización SVD con el máximo número
#    de componentes y por lo tanto el máximo de valores singulares posibles de acuerdo
#    a este criterio. Así, de acuerdo al resultado obtenido en el Ejercicio-3, asigna
#    a la variable "nc_food" el número entero adecuado para llegar a obtener el
#    resultado pedido.


# ************* Inlcuye aquí tu código:**********************************************

#Aplica Factorización SVD
U,s, Vt=np.linalg.svd(UtMx_food,full_matrices=False)
nc_food = min(UtMx_food.shape)    # número de componentes

# *********** Aquí termina la sección de agregar código *****************************

print('Total de valores singulares basados en evaluación de la comida:', nc_food)

Total de valores singulares basados en evaluación de la comida: 129


In [ ]:
# b. Lleva a cabo la factorización SVD truncada en la matriz de utilidad
#    obtenida previamente y con respecto al número de componentes dada
#    por la variable nc_food.

# ************* Inlcuye aquí tu código:********************************

nc_food=s.shape[0]
SVD_food = TruncatedSVD(n_components=nc_food, random_state=1)
SVD_food.fit_transform(UtMx_food)  #transforma

# *********** Aquí termina la sección de agregar código ***************

# Con la factorización SVD truncada obtenida, determinemos la cantidad
# de componentes que explican un 90% de la variabilidad acumulada de
# cada componente:

for j in range(nc_food):
  if SVD_food.explained_variance_ratio_[0:j].sum() > 0.90:
    break

# Usaremos esta cantidad de componentes para las recomendaciones basadas
# en la calificación de la comida:
N_food = j-1

print('Valor de truncamiento al 90% de dicha variabilidad:', N_food)

Valor de truncamiento al 90% de dicha variabilidad: 51


In [ ]:
# Usando la métrica de correlación de Pearson, obtenemos ahora las 10 mejores
# recomendaciones de este modelo no supervisado con base a la información de
# alguien que desea obtener las similitudes con el "Restaurante Pueblo Bonito"
# y con la cota de truncamiento obtenido en el ejercicio anterior.
# Para ello obtenemos las 10 mejores correlaciones positivas a continuación.

# Factorización SVD:
SVD_food = TruncatedSVD(n_components = N_food)
resultant_matrix_food = SVD_food.fit_transform(UtMx_food)

# Matriz de correlación de Pearson:
corr_mat_food = np.corrcoef(resultant_matrix_food)

# Restaurante de referencia:
restaurante_de_referencia = "Restaurante Pueblo Bonito"
nombres_rest = UtMx_food.T.columns  # nombres de restaurantes
idx_rest = list(nombres_rest).index(restaurante_de_referencia)
corr_rest = corr_mat_food[idx_rest] # Vector de Correlación del RinconHuasteco

# Buscando las correlaciones positivas:
idx = (corr_rest>0)
mejores_sim_food = list()
for i in range(len(nombres_rest[idx])):
  mejores_sim_food.append((corr_rest[idx][i], nombres_rest[idx][i]))

print('Total de similaridades positivas encontradas:', len(mejores_sim_food))

print('Algunos de los resultados encontrados:')
mejores_sim_food[0:5]

Total de similaridades positivas encontradas: 69
Algunos de los resultados encontrados:


[(np.float64(0.049896329346158716), 'Cabana Huasteca'),
 (np.float64(0.019237691327380547), 'Cafe Chaires'),
 (np.float64(0.00690516968419646), 'Cafeteria cenidet'),
 (np.float64(0.23973930486311446), 'Cafeteria y Restaurant El Pacifico'),
 (np.float64(0.03216703858465535), 'Cenaduria El RincÃ³n de Tlaquepaque')]

## **Ejercicio - 5**

In [ ]:
# Ordenar la lista de recomendaciones "mejores_sim_food" encontrada en el paso
# anterior de manera descendente. Llamaremos "mejores_sim_food_ordenadas" a
# dicha lista.



mejores_sim_food_ordenadas = sorted(mejores_sim_food, key=lambda simil: simil[0], reverse=True)



# Desplegamos las 10 mejores similitudes encontradas de manera descendente:
print('Similitudes con base a la evaluación de la comida con mayores valores de correlación:')
for k in range(1,11):
  print('%d> %s' % (k, mejores_sim_food_ordenadas[k]))

Similitudes con base a la evaluación de la comida con mayores valores de correlación:
1> (np.float64(0.7933462480897546), 'Restaurante la Estrella de Dima')
2> (np.float64(0.7093455473520455), 'pizza clasica')
3> (np.float64(0.6549111632075806), 'Restaurante Guerra')
4> (np.float64(0.5580606215005556), 'El Club')
5> (np.float64(0.38114018024167756), 'Pizzeria Julios')
6> (np.float64(0.31394011618500023), 'Hamburguesas saul')
7> (np.float64(0.30375953026366587), 'Restaurant Oriental Express')
8> (np.float64(0.29445541109292495), 'La Fontana Pizza Restaurante and Cafe')
9> (np.float64(0.2939348180965742), 'la Cochinita Pibil Restaurante Yucateco')
10> (np.float64(0.2901860559522031), 'Restaurante El Cielo Potosino')


## **Ejercicio - 6**

### **De manera análoga y usando ahora la evaluación de un restaurante con respecto a su servicio (service_rating), encuentra ahora los diez restaurantes con mayor similaridad al mismo restaurante anterior llamado "Restaurante Pueblo Bonito" y con respecto también a la misma métrica de similaridad de Pearson.**

### **Deberás usar la cantidad de componentes necesarias para que la varabilidad de la factorización SVD quede explicada en un 90%.**

### **a.  Para ello define primero la matriz de utilidad, UtMx_service, donde los renglones son los nombres de los restaurantes, las columnas los usuarios y las entradas las evaluaciones con respecto al servicio recibido en el restaurante.**





In [ ]:
# a. Define la matriz de utilidad cuyos renglones sean los nombres de los
#    restaurantes, las columnas los IDs de los usuarios y las entradas la
#    evaluación del servicio (service_rating). La llamaremos "UtMx_service".

# ************* Inlcuye aquí tu código:*****************************
UtMx_service = df_combinado.pivot_table(values="service_rating", index="name", columns="userID", fill_value=0)

# *********** Aquí termina la sección de agregar código *************

print('Dimensión de la matriz de Utilidad sobre la evaluación del servicio:')
print('(restaurantes, usuarios) =', (UtMx_service.shape))
UtMx_service.head()

Dimensión de la matriz de Utilidad sobre la evaluación del servicio:
(restaurantes, usuarios) = (129, 138)


userID,U1001,U1002,U1003,U1004,U1005,U1006,U1007,U1008,U1009,U1010,...,U1129,U1130,U1131,U1132,U1133,U1134,U1135,U1136,U1137,U1138
name,,,,,,,,,,,,,,,,,,,,,
Abondance Restaurante Bar,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Arrachela Grill,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cabana Huasteca,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
Cafe Chaires,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cafeteria cenidet,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# b. Incluye el código necesario para obtener y desplegar los 10 restaurantes con
#    mayor similaridad de Pearson al "Restaurante Pueblo Bonito", y usando solo
#    el 90% de las componentes que explican la varianza de la matriz UtMx_service.
#    Desplegarlos en orden descendente, sin incliur el restaurante de referencia.

# ************* Inlcuye aquí tu código:*****************************
 # Incluye todas las líneas de código o celdas que requieras.

# SVD completo para el máximo de componentes
U_s, s_s, Vt_s = np.linalg.svd(UtMx_service, full_matrices=False)
nc_service_max = s_s.shape[0]  # max de valores singulares

# TruncatedSVD con el max para varianza
SVD_service_full = TruncatedSVD(n_components=nc_service_max - 1, random_state=1)
SVD_service_full.fit(UtMx_service)

# buscar >= 90% de varianza explicada
varianza_acumulada = np.cumsum(SVD_service_full.explained_variance_ratio_)
N_service = np.argmax(varianza_acumulada >= 0.90) + 1

print('número de componentes para >= 90% de varianza :', N_service)

# *********** Aquí termina la sección de agregar código *************

número de componentes para >= 90% de varianza : 52


In [ ]:
# Factorización SVD truncada al N_service obtenido
SVD_service = TruncatedSVD(n_components=N_service, random_state=1)
resultant_mtx_service = SVD_service.fit_transform(UtMx_service)

# matriz de correlación de Pearson entre restaurantes
corr_mtx_service = np.corrcoef(resultant_mtx_service)

# establecer el restaurante pueblo bonito como el de referencia
restaurante_ref = "Restaurante Pueblo Bonito"
nombres_rest = UtMx_service.index   # nombres de restaurantes
idx_rest_service = list(nombres_rest).index(restaurante_ref)

# Vector de correlaciones del restaurante de referencia
corr_rest_service = corr_mat_service[idx_rest_service]

In [ ]:
# Solo correlaciones positivas y sin el propio restaurante
mejores_sim_service = list()
for i, nombre in enumerate(nombres_rest):
    if i != idx_rest_service and corr_rest_service[i] > 0:
        mejores_sim_service.append((corr_rest_service[i], nombre))

print('Total de similaridades positivas encontradas (servicio):', len(mejores_sim_service))

Total de similaridades positivas encontradas (servicio): 53


In [ ]:
# del de mas silimitud al que menos similitud tiene
mejores_sim_service_ordenadas = sorted(
    mejores_sim_service, key=lambda x: x[0], reverse=True
)

#lista de 10 restaurants
print('\nSimilitudes con base a la evaluación del servicio con mayores valores de correlación:')
for k in range(1, 11):
    print('%d> %s' % (k, mejores_sim_service_ordenadas[k-1]))


Similitudes con base a la evaluación del servicio con mayores valores de correlación:
1> (np.float64(0.6630301360540087), 'pizza clasica')
2> (np.float64(0.5805334306365726), 'El Club')
3> (np.float64(0.47770286363177733), 'Restaurante la Estrella de Dima')
4> (np.float64(0.4746663773725671), 'Hamburguesas saul')
5> (np.float64(0.42920693282536043), 'Restaurante Guerra')
6> (np.float64(0.41683862939575594), 'Restaurant Orizatlan')
7> (np.float64(0.3617482802017878), 'la Cochinita Pibil Restaurante Yucateco')
8> (np.float64(0.3500263926159478), 'Restaurante El Cielo Potosino')
9> (np.float64(0.3080681145130943), 'La Fontana Pizza Restaurante and Cafe')
10> (np.float64(0.3002785489369249), 'puesto de tacos')


## **Ejercicio - 7**

### **Incluye tus comentarios y conclusiones de la actividad. En particular indica cuáles restaurantes encontraste con correlaciones altas tanto para la calificación de la comida, como del servicio.**

++++++++ Inicia la sección de agregar texto: +++++++++++


###**Conclusión**
Durante el desarrollo de esta actividad, fue posible observar como la reducinn de dimensionalidad conserva la informacion que nos es de mayor importancia para las preferencias de los usuarios.
Se logro obtner las variables mas cercanaas al restaurante de referencia al truncar la variabilidad al 90% o mas, reduciendo increblemente rapido al exlcuir los valores (restaurantes) que no nos eran para nada utiles de el dataset original.

Al contrastar las recomendaciones a partir de una referencia, en este caso, Restaurante Pueblo Bonito, se logro encontrar coinicdencias significativoas, usando los aspectos de comida y servicio de este restaurante, esto demuestra que ambos aspectos suelen estar correlacionada en lo que seria la experiencia del cliente.

Esta práctica me parecio muy util para aplicar en cualquier sistema donde se tiene correlacion guiandose con un registro esepecifico, el lograr encontrar coincidencias con un solo objetivo es muy util para reducir el tiempo y optimizar las busquedas

###**Restaurantes identificadas con Correlación Alta con Restaurante Pueblo Bonito**

| # | Nombre | Correlación Comida | Correlación Servicio | Promedio |
|---|---|---|---|---|
| 1 | pizza clasica | 0.7093 | 0.6630 | 0.6861 |
| 2 | Restaurante la Estrella de Dima | 0.7933 | 0.4777 | 0.6355 |
| 3 | El Club | 0.5580 | 0.5805 | 0.5692 |
| 4 | Restaurante Guerra | 0.6549 | 0.4292 | 0.5420 |
| 5 | Hamburguesas saul | 0.3139 | 0.4746 | 0.3942 |
| 6 | la Cochinita Pibil Restaurante Yucateco | 0.2939 | 0.3617 | 0.3278 |
| 7 | Restaurante El Cielo Potosino | 0.2901 | 0.3500 | 0.3200 |
| 8 | La Fontana Pizza Restaurante and Cafe | 0.2944 | 0.3080 | 0.3012 |





++++++++ Termina la sección de agregar texto. +++++++++++

>> **Fin de la Actividad de Sistemas de Recomendación**